# B6. Huấn luyện chính trên Google Colab

Notebook sinh tự động bởi `ml/scripts/make_notebooks.py`. Kế hoạch: `docs/nhan-dang-robot/00-ke-hoach.md`, mục B6.

**Trước khi chạy**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Trong Google Drive phải có thư mục `robotica-ml` chứa `images-raw.zip` (ảnh của bộ dữ liệu).

**Cách chạy**: Runtime → Run all, rồi **để nguyên tab này mở** (Colab miễn phí ngắt khi đóng tab hoặc lâu không tương tác). Colab chỉ làm phần việc được giao trong ô `ARCH, SEEDS` (mặc định ViT-B/16 hạt giống 43, khoảng 3 giờ); máy nhà và Kaggle làm phần còn lại (`ml/scripts/dieu_phoi.py` tự chép kết quả từ Drive về máy nhà). Đừng đổi sang `--all`: sẽ chạy trùng việc của máy khác.

Mỗi epoch lưu checkpoint vào Drive `robotica-ml/runs/b06/`. **Nếu Colab ngắt giữa chừng**: mở lại notebook, Run all lần nữa, việc huấn luyện chạy tiếp từ epoch đã xong. Nếu không chạy lại được, máy nhà tự chạy tiếp từ checkpoint trên Drive.

Ảnh chỉ dùng cho nghiên cứu, không chia sẻ notebook kèm dữ liệu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/robotica-ml'
assert os.path.exists(f'{DRIVE}/images-raw.zip'), 'Không thấy robotica-ml/images-raw.zip trong Google Drive'
os.environ['ROBOTREC_DATA'] = '/content/data'      # ảnh giải nén ra đĩa của máy Colab: đọc nhanh hơn đọc thẳng từ Drive
os.environ['ROBOTREC_RUNS'] = f'{DRIVE}/runs'      # checkpoint và kết quả: ghi vào Drive
os.makedirs(os.environ['ROBOTREC_RUNS'], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!test -d /content/WEB_MO_HINH || git clone --depth 1 https://github.com/ledoankhoi/WEB_MO_HINH.git /content/WEB_MO_HINH
!git -C /content/WEB_MO_HINH pull --ff-only
%cd /content/WEB_MO_HINH
!git log -1 --format='Mã đang dùng: %h %s'
# Colab có sẵn PyTorch, torchvision, numpy, pandas, scipy, matplotlib; chỉ cần thêm timm (ghim đúng phiên bản đã dùng ở máy)
!pip install -q timm==1.0.31
!python -c "import torch, timm; print('torch', torch.__version__, '| timm', timm.__version__, '| GPU', torch.cuda.is_available())"

In [ ]:
# Giải nén ảnh (khoảng 3 GB, vài phút; chỉ làm một lần mỗi phiên Colab)
if not os.path.exists('/content/data/images/raw'):
    os.makedirs('/content/data/images', exist_ok=True)
    !unzip -q {DRIVE}/images-raw.zip -d /content/data/images
!python -c "import pandas as pd, os; s=pd.read_csv('ml/data/manifests/b04_splits.csv.gz'); miss=[f for f in s.file if not os.path.exists('/content/data/images/raw/'+f)]; print('ảnh của bộ chia tập:', len(s), '| thiếu:', len(miss))"

In [ ]:
# Phần việc của máy Colab này (chia việc: ml/scripts/dieu_phoi.py). Chạy tiếp được nếu bị ngắt; lần chạy đã xong được bỏ qua.
ARCH, SEEDS = 'vit_b16_clip', [43]
for seed in SEEDS:
    !python ml/scripts/b06_huan_luyen.py --arch {ARCH} --seed {seed}

In [ ]:
# Kết quả (máy nhà tự chép từ Drive về và làm báo cáo B06)
import json
for seed in SEEDS:
    path = f"{os.environ['ROBOTREC_RUNS']}/b06/{ARCH}-s{seed}/result.json"
    if os.path.exists(path):
        best = json.load(open(path, encoding='utf-8'))['best']
        print(f"{ARCH} hạt giống {seed}: xong, epoch tốt nhất {best['epoch']}, val top-1 {100 * best['val_top1']:.2f}%, macro-F1 {100 * best['val_macro_f1']:.2f}%")
    else:
        print(f'{ARCH} hạt giống {seed}: chưa xong')